In [7]:
from datetime import date, timedelta
from pathlib import Path
import polars as pl

base_dir = Path("/Volumes/T2-SSD/planequery/data/raw/adsblol/parquet_output/v6")

start_date = date(2026, 1, 1)
end_date = date(2026, 6, 20)

paths = []

d = start_date
dfs = []
while d <= end_date:
    path = (
        base_dir
        / f"year={d.year}"
        / f"month={d.month:02d}"
        / f"day={d.day:02d}"
        / "icao_bucket=5"
        / "data.parquet"
    )

    if path.exists():
        paths.append(path)
        df = pl.read_parquet(path, columns=["icao", "time", "pia", "callsign"])
        dfs.append(df)

    d += timedelta(days=1)

dfs

[shape: (3_579_038, 4)
 ┌────────┬─────────────────────────┬───────┬──────────┐
 │ icao   ┆ time                    ┆ pia   ┆ callsign │
 │ ---    ┆ ---                     ┆ ---   ┆ ---      │
 │ str    ┆ datetime[ms]            ┆ bool  ┆ str      │
 ╞════════╪═════════════════════════╪═══════╪══════════╡
 │ a0001a ┆ 2026-03-01 16:43:56.720 ┆ false ┆ CXF1     │
 │ a0001a ┆ 2026-03-01 16:43:58.090 ┆ false ┆ null     │
 │ a0001a ┆ 2026-03-01 16:44:04.450 ┆ false ┆ null     │
 │ a0001a ┆ 2026-03-01 16:44:17.090 ┆ false ┆ null     │
 │ a0001a ┆ 2026-03-01 16:44:27.070 ┆ false ┆ CXF1     │
 │ …      ┆ …                       ┆ …     ┆ …        │
 │ adf666 ┆ 2026-03-01 18:04:13.950 ┆ false ┆ null     │
 │ adf666 ┆ 2026-03-01 18:04:19.850 ┆ false ┆ N999ZW   │
 │ adf666 ┆ 2026-03-01 18:04:20.280 ┆ false ┆ null     │
 │ adf666 ┆ 2026-03-01 18:04:30.770 ┆ false ┆ null     │
 │ adf666 ┆ 2026-03-01 18:04:39.510 ┆ false ┆ null     │
 └────────┴─────────────────────────┴───────┴──────────┘,
 shape:

In [8]:
df = pl.concat(dfs)

In [18]:
df = df.filter(pl.col("pia"))
df = df.filter(
    pl.col("callsign").str.contains(r"^N[0-9][0-9A-Z]{0,4}$")
)

In [19]:
unique_callsigns = df.unique(subset=["callsign"], keep="first")
unique_callsigns

icao,time,pia,callsign
str,datetime[ms],bool,str
"""a03ed9""",2026-04-30 16:20:32.990,true,"""N247LC"""
"""a130a7""",2026-05-31 23:45:32.440,true,"""N78HR"""
"""a1011f""",2026-05-24 07:18:37.500,true,"""N52202"""
"""a03bff""",2026-05-22 22:03:32.510,true,"""N17HA"""
"""a03b55""",2026-06-02 00:51:43.360,true,"""N114CV"""
…,…,…,…
"""a091f4""",2026-04-30 14:38:32.360,true,"""N136EW"""
"""a13217""",2026-05-28 15:01:42.960,true,"""N1764X"""
"""a03ce7""",2026-05-29 14:18:34.250,true,"""N114UX"""


In [14]:
icaos = df.get_column("icao").unique().to_list()
len(icaos)

51457

NameError: name 'icao' is not defined